# corrector_07 - E5 (overlap split) and E3 (scikit-learn baseline), role B

Needs: `corrector_06_final.ipynb` already run (models in `checkpoints/corrector/`) and
`outputs/ocr_out_beam.jsonl`. **CPU runtime is enough.** Needs the `GITHUB_TOKEN` secret.

* **E5.** Flag each of the 636 OCR val lines when its label shares at least one 5-word window
  with the corrector's corpus (the first 300,000 sentences of `train.txt`, the tagger's training
  text), with the normalization of `scripts/check_text_overlap.py`. Then CER before/after and
  fixed / broken / net for each group, default level (tagger), tau = 0.
* **E3.** Logistic regression (scikit-learn) that scores the same candidates (at most 8 accented
  forms), trained on the same synthetic noise from the corpus, calibrated like the other levels,
  and scored on the same 636 lines with the same counting as Tables 8 and 9.

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

token = userdata.get("GITHUB_TOKEN").strip()
BRANCH = "b/corrector"  # change to "main" after the code is merged
!rm -rf /content/repo
!git clone -q -b {BRANCH} https://{token}@github.com/wickyhien18/Vietnamese-Handwriting-OCR.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

DRIVE_ROOT = "/content/drive/MyDrive/Vietnamese-Handwriting-OCR"
DATA_DIR = f"{DRIVE_ROOT}/data"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
OUT_DIR = f"{DRIVE_ROOT}/outputs"

In [ ]:
!python -m unittest discover -s tests -t . 2>&1 | tail -4

In [ ]:
import hashlib, itertools, json, os, time
import torch
from src.corrector import api
from src.corrector.ngram import NgramModel
from src.corrector.ocr_adapt import evaluate_on_ocr, ocr_lines, print_ocr_table

OCR_FILE = f"{OUT_DIR}/ocr_out_beam.jsonl"
MODEL_DIR = f"{CKPT_DIR}/corrector"
CORPUS_DIR = f"{DATA_DIR}/corpus"
CORPUS_LINES = 300_000          # E5: the corpus the tagger was trained on
WINDOW = 5
N_SKLEARN_TRAIN = 200_000       # E3: training sentences (first lines of train.txt)
N_CAL = 5_000                   # E3: corpus val sentences for the temperature, as for the other levels
TAUS = [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0]
device = "cuda" if torch.cuda.is_available() else "cpu"

for name in ("ocr_out.jsonl", "ocr_out_beam.jsonl"):
    p = f"{OUT_DIR}/{name}"
    if os.path.exists(p):
        print(name, hashlib.md5(open(p, "rb").read()).hexdigest())
with open(OCR_FILE, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]
val_rows = [r for r in rows if r["split"] == "val"]
lines = ocr_lines(rows, split="val")
assert len(lines) == 636, len(lines)
ngram = NgramModel.load(f"{MODEL_DIR}/ngram.pkl.gz")
tagger = api.build_corrector("tagger", MODEL_DIR, device)
print("val lines:", len(lines), "| default level:", api.default_level(MODEL_DIR), "| device:", device)

## E5. Lines with / without a 5-word window shared with the corpus

In [ ]:
from src.corrector.overlap_split import overlap_flags, print_split, split_report

flags, overlap_stats = overlap_flags([gt for gt, _ in lines], f"{CORPUS_DIR}/train.txt",
                                     window=WINDOW, corpus_lines=CORPUS_LINES)
print("check_text_overlap summary on these lines:", json.dumps(overlap_stats))
e5 = split_report(tagger, lines, flags, tau=0.0)
all_lines = evaluate_on_ocr(tagger, lines, tau=0.0, n_examples=0)
print()
print_split(e5)
c = all_lines["counts"]
print(f"{'all lines':<12}{len(lines):>6}{all_lines['cer_before']:>14.2%}{all_lines['cer_after']:>11.2%}"
      f"{c.get('fixed', 0):>7}{c.get('broken', 0):>8}{all_lines['net_fixed']:>6}  (check against Table 8)")

os.makedirs(OUT_DIR, exist_ok=True)
with open(f"{OUT_DIR}/corrector_e5_overlap_split.json", "w", encoding="utf-8") as f:
    json.dump({"ocr_file": OCR_FILE, "level": "tagger", "tau": 0.0, "window": WINDOW,
               "corpus": f"first {CORPUS_LINES} lines of data/corpus/train.txt",
               "flag": "label shares at least one 5-word window with the corpus (normalized as check_text_overlap.py)",
               "overlap_summary": overlap_stats, "groups": e5,
               "all_lines": {k: all_lines[k] for k in ("cer_before", "cer_after", "net_fixed")},
               "lines": [{"id": r["id"], "overlap": f[0], "windows": f[1], "matched": f[2]}
                         for r, f in zip(val_rows, flags)]}, f, ensure_ascii=False, indent=1)
print("saved", f"{OUT_DIR}/corrector_e5_overlap_split.json")

## E3. scikit-learn baseline: train and calibrate

In [ ]:
from src.corrector.calibration import calibration_report, fit_temperature
from src.corrector.experiments import collect_level_logits, read_lines
from src.corrector.noise import NoiseConfig, make_pairs
from src.corrector.sklearn_baseline import HYPERPARAMETERS, save, train_sklearn_baseline

start = time.time()
train_pairs = make_pairs(read_lines(f"{CORPUS_DIR}/train.txt", 0, N_SKLEARN_TRAIN), seed=1)
baseline, n_rows = train_sklearn_baseline(ngram, train_pairs, seed=0)
train_seconds = time.time() - start
print(f"trained on {N_SKLEARN_TRAIN} sentences, {n_rows} candidate rows, {train_seconds:.0f} s")

cal_pairs = list(make_pairs(read_lines(f"{CORPUS_DIR}/val.txt", 0, N_CAL), seed=123))
logits, targets = collect_level_logits(baseline, cal_pairs, HYPERPARAMETERS["k"] + 1)
baseline.temperature = fit_temperature(logits, targets)
sk_calibration = calibration_report(logits, targets, baseline.temperature)
print(json.dumps(sk_calibration, indent=1))
save(baseline, f"{MODEL_DIR}/sklearn_baseline.pkl")

## E3. Scored like Tables 8 and 9 (same 636 lines, same counting)

In [ ]:
from src.corrector.evaluate import evaluate

corpus_eval = evaluate(baseline, cal_pairs[:2000], ngram)
print("corpus val (2,000 sentences, mixed noise):",
      {k: round(corpus_eval[k], 4) for k in ("all", "ambiguous", "noisy", "broken")})

bigram = api.build_corrector("bigram", MODEL_DIR, device)
table8 = {name: evaluate_on_ocr(c, lines, tau=0.0, n_examples=0)
          for name, c in (("none", api.build_corrector("identity")), ("bigram", bigram),
                          ("tagger", tagger), ("sklearn", baseline))}
print("\nTable 8 (tau = 0):")
print_ocr_table(table8)

sweep = {f"sklearn tau={t}": evaluate_on_ocr(baseline, lines, tau=t, n_examples=0) for t in TAUS}
print("\nTable 9 (tau sweep, sklearn):")
print_ocr_table(sweep)

strip = lambda r: {k: v for k, v in r.items() if k != "examples"}
with open(f"{OUT_DIR}/corrector_e3_sklearn.json", "w", encoding="utf-8") as f:
    json.dump({"ocr_file": OCR_FILE, "model": HYPERPARAMETERS, "train_sentences": N_SKLEARN_TRAIN,
               "train_rows": n_rows, "train_seconds": round(train_seconds, 1),
               "noise": NoiseConfig().__dict__, "calibration": sk_calibration,
               "corpus_val": corpus_eval,
               "table8_tau0": {k: strip(v) for k, v in table8.items()},
               "table9_sweep": {k: strip(v) for k, v in sweep.items()}}, f, ensure_ascii=False, indent=1)
print("\nsaved", f"{OUT_DIR}/corrector_e3_sklearn.json")